<a href="https://colab.research.google.com/github/felimaker/IBM-Data-Science-Capstone/blob/main/2_web_scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 2: Recolección de Datos - Web Scraping (Wikipedia)
**Proyecto:** Predicción de aterrizaje de la primera etapa del Falcon 9

**Objetivo:** Extraer la tabla histórica de lanzamientos del Falcon 9 desde la página
de Wikipedia *"List of Falcon 9 and Falcon Heavy launches"* mediante web scraping con
`BeautifulSoup`, como fuente de datos complementaria/de verificación a la API de SpaceX.

**Diagrama de flujo:**

```
GET Wikipedia "List of Falcon 9 and Falcon Heavy launches"
      │
      ▼
Parsear HTML con BeautifulSoup
      │
      ▼
Localizar todas las tablas <table class="wikitable">
      │
      ▼
Identificar encabezados de columna válidos (extract_column_from_header)
      │
      ▼
Iterar filas <tr> de cada tabla y extraer:
  fecha, versión del cohete, sitio de lanzamiento,
  carga útil, masa, órbita, cliente, resultado del lanzamiento,
  resultado del aterrizaje de la primera etapa
      │
      ▼
Construir diccionario -> DataFrame -> dataset_part_2.csv
```


In [1]:
!pip install beautifulsoup4 requests -q

import requests
import pandas as pd
from bs4 import BeautifulSoup
import re
import unicodedata


## 1. Solicitud GET a la página de Wikipedia

In [3]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

# Agregamos un User-Agent para que Wikipedia acepte la petición
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

response = requests.get(static_url, headers=headers)
print("Status code:", response.status_code)

soup = BeautifulSoup(response.text, 'html.parser')
print(soup.title.string)


Status code: 200
List of Falcon 9 and Falcon Heavy launches - Wikipedia


## 2. Extraer todas las tablas de tipo `wikitable`

In [4]:
html_tables = soup.find_all('table', class_='wikitable plainrowheaders collapsible')
print("Número de tablas encontradas:", len(html_tables))

# La tercera tabla (índice 2) contiene el historial real de lanzamientos
first_launch_table = html_tables[2]


Número de tablas encontradas: 9


## 3. Función auxiliar: limpiar encabezados de columna

In [5]:
def extract_column_from_header(row):
    if row.br:
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
    column_name = ' '.join(row.contents)
    if not column_name.strip().isdigit():
        column_name = column_name.strip()
        return column_name

column_names = []
for th in first_launch_table.find_all('th'):
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)

print(column_names)


['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


## 4. Construir el diccionario de datos iterando filas `<tr>`

In [6]:
launch_dict = dict.fromkeys(column_names)
del launch_dict['Date and time ( )']

launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []
launch_dict['Version Booster'] = []
launch_dict['Booster landing'] = []
launch_dict['Date'] = []
launch_dict['Time'] = []

def date_time(table_cells):
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    out = ''.join([booster_version for i, booster_version in enumerate(table_cells.strings) if i % 2 == 0][0:-1])
    return out

def landing_status(table_cells):
    return [i for i in table_cells.strings][0]

def get_mass(table_cells):
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass = mass[0:mass.find("kg") + 2]
    else:
        mass = 0
    return mass

extracted_row = 0
for table_number, table in enumerate(soup.find_all('table', class_='wikitable plainrowheaders collapsible')):
    for rows in table.find_all("tr"):
        if rows.th and rows.th.string:
            flight_number = rows.th.string.strip()
            flag = flight_number.isdigit()
        else:
            flag = False

        row = rows.find_all('td')
        if flag:
            extracted_row += 1
            launch_dict['Flight No.'].append(flight_number)

            datatimelist = date_time(row[0])
            launch_dict['Date'].append(datatimelist[0].strip(','))
            launch_dict['Time'].append(datatimelist[1] if len(datatimelist) > 1 else None)

            bv = booster_version(row[1])
            if not bv:
                bv = row[1].a.string if row[1].a else None
            launch_dict['Version Booster'].append(bv)

            launch_site = row[2].a.string if row[2].a else None
            launch_dict['Launch site'].append(launch_site)

            payload = row[3].a.string if row[3].a else None
            launch_dict['Payload'].append(payload)

            payload_mass = get_mass(row[4])
            launch_dict['Payload mass'].append(payload_mass)

            orbit = row[5].a.string if row[5].a else None
            launch_dict['Orbit'].append(orbit)

            customer = row[6].a.string if row[6].a else None
            launch_dict['Customer'].append(customer)

            launch_outcome = list(row[7].strings)[0] if row[7].strings else None
            launch_dict['Launch outcome'].append(launch_outcome)

            booster_landing = landing_status(row[8]) if len(row) > 8 else None
            launch_dict['Booster landing'].append(booster_landing)

print("Filas extraídas:", extracted_row)


Filas extraídas: 121


## 5. Convertir a DataFrame y exportar

In [8]:
df_scrape = pd.DataFrame({k: pd.Series(v) for k, v in launch_dict.items()})

# Guardamos el archivo directamente en el directorio actual de Colab
df_scrape.to_csv('spacex_web_scraped.csv', index=False)

print(df_scrape.shape)
df_scrape.head()


(121, 11)


,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Version Booster,Booster landing,Date,Time
0,1,CCAFS,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,F9 v1.07B0003.1,Failure,4 June 2010,18:45
1,2,CCAFS,Dragon,0,LEO,NASA,Success,F9 v1.07B0004.1,Failure,8 December 2010,15:43
2,3,CCAFS,Dragon,525 kg,LEO,NASA,Success,F9 v1.07B0005.1,No,22 May 2012,07:44
3,4,CCAFS,SpaceX CRS-1,"4,700 kg",LEO,NASA,Success,F9 v1.07B0006.1,No attempt,8 October 2012,00:35
4,5,CCAFS,SpaceX CRS-2,"4,877 kg",LEO,NASA,Success,F9 v1.07B0007.1,No,1 March 2013,15:10


## Resumen del proceso
- Fuente: página de Wikipedia (versión fija por `oldid` para reproducibilidad).
- Se localizaron las tablas `wikitable` de la página y se identificó la tabla de
  historial completo de lanzamientos.
- Se limpiaron los encabezados (eliminando referencias `<sup>`, saltos `<br>` y enlaces
  `<a>`) para obtener nombres de columna legibles.
- Se iteró cada fila `<tr>` extrayendo: número de vuelo, fecha/hora, versión del
  cohete, sitio de lanzamiento, payload, masa, órbita, cliente, resultado del
  lanzamiento y resultado del aterrizaje del booster.
- Resultado exportado a `spacex_web_scraped.csv`.
- Repositorio de GitHub: **`<https://github.com/felimaker/IBM-Data-Science-Capstone>`**
